In [1]:
!pip install anthropic --quiet

import os
import json
import anthropic
import numpy as np
import pandas as pd
from datetime import datetime
from dotenv import load_dotenv
from supabase import create_client, Client
from sklearn.ensemble import IsolationForest, GradientBoostingRegressor
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

load_dotenv(dotenv_path="/Users/mohamedmunir/Desktop/FinScope-Ai/ai-service/notebooks/.env", override=True)

supabase: Client = create_client(os.getenv("SUPABASE_URL"), os.getenv("SUPABASE_SERVICE_KEY"))
claude           = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

print("Connected to Supabase and Claude")


Connected to Supabase and Claude


In [2]:
# Tools the agent can call
TOOLS = [
    {
        "name": "pull_data",
        "description": "Pull all transactions and user finances from Supabase",
        "input_schema": {"type": "object", "properties": {}, "required": []}
    },
    {
        "name": "clean_data",
        "description": "Clean and prepare transaction data for ML models",
        "input_schema": {
            "type": "object",
            "properties": {
                "filter_type": {
                    "type": "string",
                    "description": "Filter by transaction type: expense, deposit, all",
                    "enum": ["expense", "deposit", "all"]
                }
            },
            "required": ["filter_type"]
        }
    },
    {
        "name": "run_anomaly_detection",
        "description": "Run Isolation Forest to detect anomalous transactions",
        "input_schema": {
            "type": "object",
            "properties": {
                "contamination": {
                    "type": "number",
                    "description": "Expected proportion of anomalies (0.05 to 0.3)"
                }
            },
            "required": ["contamination"]
        }
    },
    {
        "name": "run_clustering",
        "description": "Run KMeans to segment users into spending personas",
        "input_schema": {
            "type": "object",
            "properties": {
                "n_clusters": {
                    "type": "integer",
                    "description": "Number of user clusters (2 to 5)"
                }
            },
            "required": ["n_clusters"]
        }
    },
    {
        "name": "run_forecast",
        "description": "Run Gradient Boosting to forecast next 7 days of spending",
        "input_schema": {"type": "object", "properties": {}, "required": []}
    },
    {
        "name": "write_insights_to_supabase",
        "description": "Write all ML insights and recommendations back to Supabase",
        "input_schema": {
            "type": "object",
            "properties": {
                "insights": {
                    "type": "object",
                    "description": "Dict of insights to store"
                }
            },
            "required": ["insights"]
        }
    },
    {
        "name": "generate_report",
        "description": "Generate a final natural language report summarising all findings",
        "input_schema": {
            "type": "object",
            "properties": {
                "findings": {
                    "type": "object",
                    "description": "All ML findings to summarise"
                }
            },
            "required": ["findings"]
        }
    }
]

print(f"Defined {len(TOOLS)} agent tools")

# git commit -m "# Tools the agent can call
TOOLS = [
    {
        "name": "pull_data",
        "description": "Pull all transactions and user finances from Supabase",
        "input_schema": {"type": "object", "properties": {}, "required": []}
    },
    {
        "name": "clean_data",
        "description": "Clean and prepare transaction data for ML models",
        "input_schema": {
            "type": "object",
            "properties": {
                "filter_type": {
                    "type": "string",
                    "description": "Filter by transaction type: expense, deposit, all",
                    "enum": ["expense", "deposit", "all"]
                }
            },
            "required": ["filter_type"]
        }
    },
    {
        "name": "run_anomaly_detection",
        "description": "Run Isolation Forest to detect anomalous transactions",
        "input_schema": {
            "type": "object",
            "properties": {
                "contamination": {
                    "type": "number",
                    "description": "Expected proportion of anomalies (0.05 to 0.3)"
                }
            },
            "required": ["contamination"]
        }
    },
    {
        "name": "run_clustering",
        "description": "Run KMeans to segment users into spending personas",
        "input_schema": {
            "type": "object",
            "properties": {
                "n_clusters": {
                    "type": "integer",
                    "description": "Number of user clusters (2 to 5)"
                }
            },
            "required": ["n_clusters"]
        }
    },
    {
        "name": "run_forecast",
        "description": "Run Gradient Boosting to forecast next 7 days of spending",
        "input_schema": {"type": "object", "properties": {}, "required": []}
    },
    {
        "name": "write_insights_to_supabase",
        "description": "Write all ML insights and recommendations back to Supabase",
        "input_schema": {
            "type": "object",
            "properties": {
                "insights": {
                    "type": "object",
                    "description": "Dict of insights to store"
                }
            },
            "required": ["insights"]
        }
    },
    {
        "name": "generate_report",
        "description": "Generate a final natural language report summarising all findings",
        "input_schema": {
            "type": "object",
            "properties": {
                "findings": {
                    "type": "object",
                    "description": "All ML findings to summarise"
                }
            },
            "required": ["findings"]
        }
    }
]

print(f"Defined {len(TOOLS)} agent tools")



Defined 7 agent tools


In [8]:
def run_pipeline_agent():
    print("\n" + "=" * 60)
    print("  FINSCOPE AI — DATA PIPELINE AGENT")
    print("  Powered by Claude + Scikit-learn + Supabase")
    print("=" * 60)

    messages = [{
        "role": "user",
        "content": """You are an autonomous data pipeline agent for FinScope AI.

Your job is to run the full ML pipeline in this exact order:
1. Pull all data from Supabase
2. Clean the expense transactions
3. Run anomaly detection with appropriate contamination
4. Run user clustering with the right number of clusters
5. Run the spending forecast
6. Write all insights back to Supabase
7. Generate a professional report of all findings

Run every step autonomously. Do not ask for confirmation.
Use your judgment on parameters based on what the data tells you."""
    }]

    findings    = {}
    step        = 0
    max_steps   = 20

    while step < max_steps:
        step += 1
        print(f"\n--- Agent Step {step} ---")

        response = claude.messages.create(
            model      = "claude-haiku-4-5-20251001",
            max_tokens = 1024,
            tools      = TOOLS,
            messages   = messages
        )

        print(f"Stop reason: {response.stop_reason}")

        # Agent finished
        if response.stop_reason == "end_turn":
            for block in response.content:
                if hasattr(block, "text"):
                    print(f"\nAgent: {block.text}")
            break

        # Agent wants to use a tool
        if response.stop_reason == "tool_use":
            messages.append({"role": "assistant", "content": response.content})
            tool_results = []

            for block in response.content:
                if block.type == "tool_use":
                    result = execute_tool(block.name, block.input)
                    print(f"  Result: {json.dumps(result, indent=2)[:300]}")

                    # Accumulate findings
                    if block.name == "run_anomaly_detection": findings["anomalies"] = result
                    if block.name == "run_clustering":        findings["clusters"]  = result
                    if block.name == "run_forecast":          findings["forecast"]  = result

                    tool_results.append({
                        "type":        "tool_result",
                        "tool_use_id": block.id,
                        "content":     json.dumps(result)
                    })

            messages.append({"role": "user", "content": tool_results})

    print("\n" + "=" * 60)
    print("  PIPELINE COMPLETE")
    print("=" * 60)

    if "report" in agent_state:
        print("\n" + agent_state["report"])

    return agent_state

# RUN IT
state = run_pipeline_agent()

# git commit -m "ml: autonomous claude agent runs full pipeline end-to-end"


  FINSCOPE AI — DATA PIPELINE AGENT
  Powered by Claude + Scikit-learn + Supabase

--- Agent Step 1 ---


TypeError: "Could not resolve authentication method. Expected either api_key or auth_token to be set. Or for one of the `X-Api-Key` or `Authorization` headers to be explicitly omitted"

In [7]:
load_dotenv(dotenv_path="/Users/mohamedmunir/Desktop/FinScope-Ai/ai-service/notebooks/.env", override=True)
key = os.getenv("ANTHROPIC_API_KEY")
print("Valid format:", key.startswith("sk-ant-") if key else False)
print("Length:", len(key) if key else 0)

Valid format: True
Length: 108


In [11]:
claude = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))
state  = run_pipeline_agent()


  FINSCOPE AI — DATA PIPELINE AGENT
  Powered by Claude + Scikit-learn + Supabase

--- Agent Step 1 ---
Stop reason: tool_use

  >> Executing: pull_data | input: {}
  Result: {
  "status": "success",
  "transactions": 52,
  "users": 8,
  "finances": 10,
  "date_range": "2026-02-15 to 2026-03-04"
}

--- Agent Step 2 ---
Stop reason: tool_use

  >> Executing: clean_data | input: {'filter_type': 'expense'}
  Result: {
  "status": "success",
  "rows_cleaned": 31,
  "categories": {
    "Subscriptions": 16,
    "Shopping": 5,
    "Bills": 3,
    "Dining": 2,
    "Transport": 2,
    "Other": 2,
    "Groceries": 1
  },
  "nulls_removed": 21
}

--- Agent Step 3 ---
Stop reason: tool_use

  >> Executing: run_anomaly_detection | input: {'contamination': 0.1}
  Result: {
  "status": "success",
  "total_expenses": 31,
  "anomalies_found": 3,
  "anomaly_rate_pct": 9.7,
  "flagged_users": 1,
  "highest_anomaly": {
    "amount": 120000.0,
    "category": "Other"
  }
}

  >> Executing: run_clustering 

/Users/mohamedmunir/Desktop/FinScope-Ai/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but GradientBoostingRegressor was fitted with feature names
  warnings.warn(


Stop reason: tool_use

  >> Executing: write_insights_to_supabase | input: {'insights': {'pipeline_status': 'complete', 'data_summary': {'total_transactions': 52, 'total_users': 8, 'total_finances_records': 10, 'date_range': '2026-02-15 to 2026-03-04'}, 'data_cleaning': {'rows_cleaned': 31, 'nulls_removed': 21, 'categories': {'Subscriptions': 16, 'Shopping': 5, 'Bills': 3, 'Dining': 2, 'Transport': 2, 'Other': 2, 'Groceries': 1}}, 'anomaly_detection': {'total_expenses_analyzed': 31, 'anomalies_found': 3, 'anomaly_rate_pct': 9.7, 'contamination_used': 0.1, 'flagged_users': 1, 'highest_anomaly': {'amount': 120000.0, 'category': 'Other', 'severity': 'critical'}}, 'user_clustering': {'n_clusters': 3, 'inertia': 5.25, 'spending_personas': {'Saver': {'user_count': 2, 'avg_savings_rate': 0.787}, 'Lifestyle Spender': {'user_count': 1, 'avg_savings_rate': 0.98}, 'Essentials First': {'user_count': 1, 'avg_savings_rate': 0.989}}}, 'spending_forecast': {'forecast_period': '7_days', 'average_daily_

/var/folders/_l/vdgqbw4x0q99dgnnpmyn9v3m0000gn/T/ipykernel_4057/2041119715.py:124: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "run_at":     datetime.utcnow().isoformat(),


Stop reason: tool_use

  >> Executing: generate_report | input: {'findings': {'pipeline_status': 'complete', 'data_summary': {'total_transactions': 52, 'total_users': 8, 'total_finances_records': 10, 'date_range': '2026-02-15 to 2026-03-04'}, 'data_cleaning': {'rows_cleaned': 31, 'nulls_removed': 21, 'categories': {'Subscriptions': 16, 'Shopping': 5, 'Bills': 3, 'Dining': 2, 'Transport': 2, 'Other': 2, 'Groceries': 1}}, 'anomaly_detection': {'total_expenses_analyzed': 31, 'anomalies_found': 3, 'anomaly_rate_pct': 9.7, 'contamination_used': 0.1, 'flagged_users': 1, 'highest_anomaly': {'amount': 120000.0, 'category': 'Other', 'severity': 'critical'}}, 'user_clustering': {'n_clusters': 3, 'inertia': 5.25, 'spending_personas': {'Saver': {'user_count': 2, 'avg_savings_rate': 0.787}, 'Lifestyle Spender': {'user_count': 1, 'avg_savings_rate': 0.98}, 'Essentials First': {'user_count': 1, 'avg_savings_rate': 0.989}}}, 'spending_forecast': {'forecast_period': '7_days', 'average_daily_forecast': 